# Phát hiện tin nhắn lừa đảo

**Sổ Nghiên Cứu AI, dự án mẫu 3/10.** Mô hình đối chứng TF-IDF, kiểm tra độ bền khi kẻ gian biến đổi tin nhắn, và khung tinh chỉnh PhoBERT cho tiếng Việt.

**Câu hỏi nghiên cứu:** Mô hình có chống chịu được khi tin nhắn bị viết không dấu, chèn ký tự hoặc teencode không?

**Giả thuyết:** Mô hình dựa trên ký tự giảm F1 ít hơn mô hình dựa trên từ khi tin nhắn bị biến đổi.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

Phần 1 luyện trên bộ **SMS Spam Collection** công khai (tiếng Anh, UCI) để có dữ liệu thật ngay. Phần 2 là quy trình cho **tiếng Việt** trên file dữ liệu em tự thu thập.

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)

import io, zipfile, re, unicodedata, requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

## Phần 1. Dữ liệu công khai

In [ ]:
def tai_sms():
    r = requests.get("https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip", timeout=60)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    t = z.read("SMSSpamCollection").decode("utf-8", "ignore").strip().split("\n")
    return pd.DataFrame([l.split("\t", 1) for l in t], columns=["nhan", "noi_dung"])
try:
    sms = tai_sms(); NGUON = "UCI SMS Spam Collection"
except Exception as e:
    print("Không tải được (", type(e).__name__, "), dùng vài câu mẫu giả lập.")
    a = ["WINNER! Claim your prize now, call 0900", "Free entry to win cash, text WIN", "Your account is locked, verify at bit.ly/x", "URGENT: you won a car, reply now"]
    b = ["See you at lunch?", "Mom said dinner at 7", "Can you send me the homework", "Ok I am on my way"]
    sms = pd.DataFrame({"nhan": ["spam"] * 200 + ["ham"] * 800, "noi_dung": [random.choice(a) + " " + str(i) for i in range(200)] + [random.choice(b) + " " + str(i) for i in range(800)]})
    NGUON = "GIẢ LẬP"
sms["y"] = (sms.nhan == "spam").astype(int)
print(NGUON, "|", len(sms), "tin | tỉ lệ lừa đảo/spam:", round(sms.y.mean(), 3))
tr, te = train_test_split(sms, test_size=0.25, stratify=sms.y, random_state=SEED)

## Hai mô hình đối chứng: theo từ và theo cụm ký tự

In [ ]:
MH = {
 "TF-IDF theo từ + logistic": make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), LogisticRegression(max_iter=2000, class_weight="balanced")),
 "TF-IDF theo ký tự + logistic": make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2), LogisticRegression(max_iter=2000, class_weight="balanced")),
}
for ten, m in MH.items():
    m.fit(tr.noi_dung, tr.y); print(ten, "F1 =", round(f1_score(te.y, m.predict(te.noi_dung)), 3))

## Bộ biến đổi đối kháng
Kẻ gian thay chữ, chèn dấu chấm, bỏ dấu để qua mặt bộ lọc. Ta tự tạo các biến đổi đó và đo mức sụt giảm.

In [ ]:
def bo_dau(s): return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn").replace("đ", "d").replace("Đ", "D")
def doi_ky_tu(s, p=0.3):
    bang = {"a": "@", "o": "0", "i": "1", "e": "3", "s": "$"}
    return "".join(bang[c] if c in bang and random.random() < p else c for c in s)
def chen_dau_cham(s, p=0.3):
    return " ".join(".".join(w) if len(w) > 3 and random.random() < p else w for w in s.split())
BIEN_DOI = {"Gốc": lambda s: s, "Bỏ dấu": bo_dau, "Đổi ký tự (a→@, o→0)": doi_ky_tu, "Chèn dấu chấm (f.r.e.e)": chen_dau_cham}
bang = {}
for ten, m in MH.items():
    bang[ten] = {bd: f1_score(te.y, m.predict(te.noi_dung.map(f))) for bd, f in BIEN_DOI.items()}
kq = pd.DataFrame(bang).round(3); print(kq)
kq.plot(kind="bar", figsize=(8, 3.5), ylim=(0, 1), title="F1 khi tin nhắn bị biến đổi"); plt.xticks(rotation=0, fontsize=8); plt.show()

## Phần 2. Tiếng Việt với dữ liệu của em
Tạo file `tin_nhan.csv` gồm hai cột `noi_dung, nhan` (`nhan` = 1 nếu lừa đảo, 0 nếu bình thường). **Ẩn danh hóa trước khi lưu:** thay số điện thoại, số tài khoản, tên người bằng `<SDT>`, `<STK>`, `<TEN>`, và có sự đồng ý của người cung cấp.

Ô dưới có sẵn vài câu mẫu do Sổ Nghiên Cứu AI tự viết, chỉ để kiểm tra code chạy được.

In [ ]:
MAU_VI = pd.DataFrame({"noi_dung": [
 "Tai khoan cua quy khach bi khoa, vui long dang nhap <LINK> de xac minh", "Chúc mừng bạn trúng thưởng xe máy, nộp phí nhận giải vào <STK>",
 "Viec nhe luong cao 500k/ngay, chi can like video, ib ngay", "Cơ quan công an thông báo bạn liên quan vụ án, gọi <SDT> gấp",
 "Bạn được hoàn tiền thuế, bấm <LINK> nhập mã OTP", "Con đang cấp cứu, mẹ chuyển gấp 5 triệu vào <STK>",
 "Tối nay 7 giờ họp lớp ở nhà văn hóa nhé", "Mẹ ơi con về muộn 30 phút", "Nhớ mang sách Toán ngày mai",
 "Cô gửi lịch kiểm tra giữa kỳ trên nhóm lớp", "Mai đi đá bóng không", "Đơn hàng của bạn đang được giao, shipper sẽ gọi"],
 "nhan": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]})
VI = pd.read_csv("tin_nhan.csv") if os.path.exists("tin_nhan.csv") else MAU_VI
print("Dữ liệu tiếng Việt:", "của em" if os.path.exists("tin_nhan.csv") else "MẪU", "|", len(VI), "tin")
m_vi = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)), LogisticRegression(max_iter=2000, class_weight="balanced"))
if len(VI) >= 100:
    a, b = train_test_split(VI, test_size=0.25, stratify=VI.nhan, random_state=SEED)
    m_vi.fit(a.noi_dung, a.nhan)
    for bd, f in {"Gốc": lambda s: s, "Bỏ dấu": bo_dau}.items():
        print(bd, "F1 =", round(f1_score(b.nhan, m_vi.predict(b.noi_dung.map(f))), 3))
else:
    m_vi.fit(VI.noi_dung, VI.nhan)
    print("Cần ít nhất 100 tin để đánh giá có ý nghĩa. Thử dự đoán:", m_vi.predict(["Ban trung thuong iPhone, nop phi tai <LINK>"]))

## Tinh chỉnh PhoBERT (chạy khi đã có từ vài trăm tin và bật GPU)
PhoBERT của VinAI được huấn luyện trên văn bản tiếng Việt đã **tách từ** (ví dụ `tài_khoản`). Để đạt kết quả tốt, tách từ trước bằng VnCoreNLP hoặc underthesea; ghi rõ cách làm trong báo cáo.

In [ ]:
CHAY_PHOBERT = False      # đổi thành True khi đã có dữ liệu thật và GPU
if CHAY_PHOBERT:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"])
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    TEN = "vinai/phobert-base"
    tok = AutoTokenizer.from_pretrained(TEN); mh = AutoModelForSequenceClassification.from_pretrained(TEN, num_labels=2).to("cuda")
    a, b = train_test_split(VI, test_size=0.25, stratify=VI.nhan, random_state=SEED)
    def mahoa(t): return tok(list(t), truncation=True, padding=True, max_length=96, return_tensors="pt")
    opt = torch.optim.AdamW(mh.parameters(), lr=2e-5)
    for ep in range(3):
        mh.train()
        for i in range(0, len(a), 16):
            lo = a.iloc[i:i + 16]; x = {k: v.to("cuda") for k, v in mahoa(lo.noi_dung).items()}
            out = mh(**x, labels=torch.tensor(lo.nhan.values).to("cuda")); out.loss.backward(); opt.step(); opt.zero_grad()
        mh.eval()
        with torch.no_grad():
            p = [mh(**{k: v.to("cuda") for k, v in mahoa(b.noi_dung.iloc[i:i + 32]).items()}).logits.argmax(1).cpu().numpy() for i in range(0, len(b), 32)]
        print("Vòng", ep + 1, "F1 =", round(f1_score(b.nhan, np.concatenate(p)), 3))
else:
    print("Bỏ qua PhoBERT. Đổi CHAY_PHOBERT = True khi sẵn sàng.")

## Việc của em
1. Thu thập ít nhất 300 tin nhắn có đồng ý, ẩn danh hóa, gán nhãn bởi 2 người và tính tỉ lệ đồng thuận.
2. Viết thêm bộ biến đổi tiếng Việt: teencode (`k` thay `không`), viết tắt, chèn emoji.
3. So sánh TF-IDF ký tự với PhoBERT trên cả tập gốc lẫn tập đã biến đổi. Đó là câu trả lời cho câu hỏi nghiên cứu.